# Comparing CodeQL CLI vs GitHub Actions scan modes for CI integration

CodeQL supports two main execution modes for repository scanning. The CodeQL CLI runs database creation and query analysis as local commands under operator control. The GitHub Actions mode runs the same analysis as managed workflow steps (init, autobuild, analyze) with results uploaded as repository alerts. This notebook compares the two modes across workflow, control, and CI-integration dimensions.

## Purpose

This notebook provides a structured comparison of CodeQL CLI scanning and GitHub Actions scanning for CI integration. It documents the command sequence for each mode, identifies where control differs (database handling, query selection, result output), and gives a decision rule for choosing between them in a delivery pipeline.

## When to Use Each Mode

### CLI mode is appropriate when:

- The scan target is not hosted on GitHub, or the pipeline runs on a different CI system.
- The operator needs direct control over database creation (custom build commands, partial extraction, or a pre-built database).
- Queries need to run locally for iteration before being committed to a shared workflow.
- Results need to stay local (file output) rather than being uploaded as repository alerts.

### GitHub Actions mode is appropriate when:

- The repository is hosted on GitHub and the team wants scanning on a schedule plus on push and pull request events.
- The workflow should stay declarative: checkout, init, autobuild, analyze, with results uploaded automatically.
- Multi-language coverage is needed through a matrix over the supported language list.
- Enforcement should come from repository branch protection and code-scanning alerts rather than from a local gate.

## Prerequisites

- A sample repository with at least one supported language (Python or JavaScript is sufficient for comparison).
- For CLI mode: the CodeQL CLI available in PATH and a buildable checkout of the target.
- For Actions mode: a GitHub-hosted repository with code scanning enabled and a workflow file under the workflows directory.
- A companion multi-language workflow exists in this kit at `codeql/manifests/multi-language-codeql-analysis.yaml` (checkout, init, autobuild, analyze with a language matrix).

In [ ]:
# last_verified: 2026-09-18 · CodeQL n/a
import shutil

for tool in ["codeql", "gh"]:
    path = shutil.which(tool)
    if path:
        print(f"OK: {tool} found at {path}")
    else:
        print(f"INFO: {tool} not in PATH -- comparison cells below still run (they model the workflows).")

## Steps

### Step 1: CLI mode command sequence

CLI scanning separates database creation from query analysis:

```bash
codeql database create codeql-db --language=python --source-root=.
codeql database analyze codeql-db --format=sarif-latest --output=results.sarif
```

The first command extracts the source into a queryable database. The second runs the selected query set against that database and writes a result file. Each stage can be re-run independently: rebuild the database when the source changes, re-analyze when the query selection changes.

In [ ]:
# last_verified: 2026-09-18 · CodeQL n/a
# Model the CLI mode as data: stages, inputs, and outputs
CLI_MODE = {
    "stages": ["database create", "database analyze"],
    "database_step": {
        "command": "codeql database create codeql-db --language=python --source-root=.",
        "input": "source checkout",
        "output": "codeql-db/ (queryable database directory)",
        "rerun_when": "source changes",
    },
    "analyze_step": {
        "command": "codeql database analyze codeql-db --format=sarif-latest --output=results.sarif",
        "input": "codeql-db/ + query selection",
        "output": "results.sarif (local result file)",
        "rerun_when": "query selection changes",
    },
    "control": "operator owns each invocation and flag",
    "result_sink": "local file",
}
print("CLI database step :", CLI_MODE["database_step"]["command"])
print("CLI analyze step  :", CLI_MODE["analyze_step"]["command"])

### Step 2: GitHub Actions mode step sequence

Actions scanning expresses the same analysis as workflow steps:

```yaml
steps:
  - uses: actions/checkout@v4
  - uses: github/codeql-action/init@v3
  - uses: github/codeql-action/autobuild@v3
  - uses: github/codeql-action/analyze@v3
```

Init selects languages and query configuration, autobuild attempts to build the project without a hand-written build command, and analyze runs the queries and uploads results. The companion workflow in this kit (`codeql/manifests/multi-language-codeql-analysis.yaml`) follows this shape with a language matrix and scheduled, push, and pull-request triggers.

In [ ]:
# last_verified: 2026-09-18 · CodeQL n/a
# Model the Actions mode as data: steps, inputs, and outputs
ACTIONS_MODE = {
    "steps": ["checkout", "init", "autobuild", "analyze"],
    "init": "selects languages and query configuration",
    "autobuild": "attempts project build without a hand-written build command",
    "analyze": "runs queries and uploads results as repository alerts",
    "control": "workflow file owns the sequence; runner owns execution",
    "result_sink": "repository code-scanning alerts",
}
print("Actions steps:", " -> ".join(ACTIONS_MODE["steps"]))
print("Result sink:", ACTIONS_MODE["result_sink"])

### Step 3: Side-by-side comparison

The following cell renders the comparison across the dimensions that matter for CI integration: trigger model, build handling, language coverage, result destination, and debuggability.

In [ ]:
# last_verified: 2026-09-18 · CodeQL n/a
# Side-by-side comparison across CI-integration dimensions
ROWS = [
    ("Trigger model", "operator runs commands when ready", "events drive runs: push, pull request, schedule"),
    ("Build handling", "explicit: operator supplies the build command", "assisted: autobuild step attempts the build"),
    ("Language coverage", "one invocation per language/database", "matrix over the language list in one workflow"),
    ("Query selection", "flags on the analyze invocation", "init-step configuration in the workflow file"),
    ("Result destination", "local result file for triage", "repository alerts with review integration"),
    ("Debuggability", "high: rerun either stage with adjusted flags", "medium: rerun via workflow re-run and step logs"),
    ("Portability", "any CI system or workstation", "GitHub-hosted repositories"),
]

print(f"{'Dimension':<20} {'CLI':<48} {'Actions'}")
print("-" * 110)
for dim, cli, actions in ROWS:
    print(f"{dim:<20} {cli:<48} {actions}")

### Step 4: Decision rule

Use CLI mode for local iteration and for non-GitHub pipelines where the operator must own database creation. Use Actions mode as the shared team gate where declarative scheduling, matrix coverage, and alert upload matter more than per-invocation control. A common combined setup is CLI for query development and Actions for the merged-code gate.

In [ ]:
# last_verified: 2026-09-18 · CodeQL n/a
def recommend(hosted_on_github, needs_local_iteration, needs_builtin_alerts):
    """Return the recommended scan mode for the given constraints."""
    if needs_local_iteration:
        return "CLI: iterate on database create/analyze locally, then commit the query selection"
    if hosted_on_github and needs_builtin_alerts:
        return "Actions: init/autobuild/analyze workflow with language matrix and alert upload"
    if hosted_on_github:
        return "Actions for the shared gate; CLI for ad-hoc local checks"
    return "CLI: portable across CI systems, results as local files"

cases = [
    ("GitHub repo, team gate", True, False, True),
    ("tuning queries locally", True, True, False),
    ("non-GitHub pipeline", False, False, False),
]
for label, gh, local, alerts in cases:
    print(f"{label:<28} -> {recommend(gh, local, alerts)}")

## Verify

To confirm the comparison holds in a given repository:

1. Run the CLI sequence against a sample checkout and confirm a local result file is produced.
2. Trigger the Actions workflow (push, pull request, or manual dispatch) and confirm the analyze step completes.
3. Confirm results appear in the expected sink: local file for CLI, repository alerts for Actions.
4. Change one query-selection input and confirm only the analyze stage needs a re-run in CLI mode.
5. Confirm the language matrix in the companion workflow covers each language the repository ships.

## Common Errors

- **Analyzing before creating the database:** the analyze stage requires a database directory from a completed create stage; running it first fails with a missing-database error.
- **Skipping the build for compiled languages:** database creation without a working build produces an empty database and empty results; fix the build command before re-running.
- **Relying on autobuild for a non-standard build:** the autobuild step covers common layouts only; repositories with custom build orchestration need an explicit manual-build replacement step.
- **Expecting local files from Actions mode:** the analyze step uploads results to the repository; looking for a local artifact on the runner misses the actual sink.
- **One database per language assumed shared:** each language needs its own database in CLI mode; reusing a Python database for a JavaScript analysis yields no findings.

## References

- CodeQL CLI documentation — database create and database analyze command reference
- CodeQL Actions documentation — init, autobuild, and analyze workflow steps
- CodeQL query documentation — default query suites and custom query selection
- Companion workflow in this kit — codeql/manifests/multi-language-codeql-analysis.yaml

In [ ]:
# last_verified: 2026-09-18 · CodeQL n/a
# Summary of the comparison
print("=" * 64)
print("CLI mode: operator-controlled, local result file, portable.")
print("Actions mode: event-driven, matrix coverage, alert upload.")
print("Combined default: CLI for iteration, Actions for the team gate.")
print("=" * 64)